# PlateHandler Main Function Walkthrough

This notebook shows the main `PlateHandler` workflow: initialize the xArm, register calibrated positions, save/load position JSON, register plates, inspect occupancy, move plates, and exchange plates.

In [16]:
from xarm.wrapper import XArmAPI

from plate_handler import PlateHandler

ARM_IP = "192.168.1.205"
POSITION_FILE = "./position.json"

## Initialize xArm and PlateHandler

`initialize()` configures the xArm for this plate-transfer setup. It does not calibrate positions by itself.

In [20]:
arm = XArmAPI(ARM_IP, do_not_open=True)
arm.connect()

plate_handler = PlateHandler(arm)
plate_handler.initialize()

ROBOT_IP: 192.168.1.205, VERSION: v2.7.0, PROTOCOL: V1, DETAIL: 6,6,XI1304,AC1303,v2.7.0, TYPE1300: [1, 1]
change protocol identifier to 3
[clean_error], xArm is ready to move
[set_state], xArm is ready to move


## Register and Save Positions

`register_position()` physically moves the robot during calibration. With `save_to_file=True`, the calibrated position is added to or updated in `POSITION_FILE`.

In [21]:
plate_handler.register_position(
    "Shelf_1",
    overwrite=True,
    save_to_file=True,
    position_file_path=POSITION_FILE,
)

Initial yaw correction: 5.711° | Fine yaw correction: 0.179°
Coarse edge detection after 160.00 mm of positive-Y travel.
Plate edge precisely detected after 155.50 mm of positive-Y travel.
Moved 145.00 mm in negative Y from the detected edge to the plate center.


{'position_id': 'Shelf_1',
 'pose': array([ 434.052399,  365.851379,  235.617661, -179.698014,   -1.189804,
          86.52207 ]),
 'loaded_plate': None}

In [22]:
plate_handler.register_position(
    "Shelf_2",
    overwrite=True,
    save_to_file=True,
    position_file_path=POSITION_FILE,
)

[set_state], xArm is ready to move
Initial yaw correction: 2.862° | Fine yaw correction: -1.790°
Coarse edge detection after 150.00 mm of positive-Y travel.
Plate edge precisely detected after 150.00 mm of positive-Y travel.
Moved 145.00 mm in negative Y from the detected edge to the plate center.


{'position_id': 'Shelf_2',
 'pose': array([ 4.40528748e+02,  3.58593811e+02,  1.67744766e+02,  1.78855422e+02,
        -2.52904000e-01,  8.91004950e+01]),
 'loaded_plate': None}

In [23]:
plate_handler.register_position(
    "Jubilee",
    overwrite=True,
    save_to_file=True,
    position_file_path=POSITION_FILE,
)

[set_state], xArm is ready to move
Initial yaw correction: 5.711° | Fine yaw correction: -2.148°
Coarse edge detection after 160.00 mm of positive-Y travel.
Plate edge precisely detected after 157.00 mm of positive-Y travel.
Moved 145.00 mm in negative Y from the detected edge to the plate center.


{'position_id': 'Jubilee',
 'pose': array([ 3.87684937e+02, -5.20455750e+02,  2.56301056e+02, -1.79117779e+02,
         2.79718000e-01, -8.53290570e+01]),
 'loaded_plate': None}

## Load Positions From JSON

In a later session/different code file, the saved calibration data can be loaded instead of recalibrating every position.

In [ ]:
plate_handler.load_positions(POSITION_FILE)
plate_handler.plate_positions

## Register Plate Occupancy

`register_plate()` only updates the software registry. It does not move the robot.

In [24]:
plate_handler.register_plate("plate_1", "Shelf_1")
plate_handler.register_plate("plate_2", "Shelf_2")

plate_handler.get_status()

{'Shelf_1': 'plate_1', 'Shelf_2': 'plate_2', 'Jubilee': None}

## Query Positions and Plates

In [25]:
# Full registered position record, including pose and loaded_plate.
plate_handler.get_position("Jubilee")

{'position_id': 'Jubilee',
 'pose': array([ 3.87684937e+02, -5.20455750e+02,  2.56301056e+02, -1.79117779e+02,
         2.79718000e-01, -8.53290570e+01]),
 'loaded_plate': None}

In [26]:
# Which position currently contains plate_1?
plate_handler.get_plate_position("plate_1")

'Shelf_1'

In [27]:
# Which plate is currently registered at Jubilee?
plate_at_jubilee = plate_handler.get_plate_at_position("Jubilee")

if plate_at_jubilee is None:
    print("Jubilee is empty")
else:
    print(f"Jubilee contains {plate_at_jubilee}")

Jubilee is empty


## Move a Plate

`move_plate()` physically transfers a registered plate from its current position to an empty destination.

In [28]:
plate_handler.move_plate(
    plate_id="plate_1",
    destination_position_id="Jubilee",
    move_speed=100,
)

plate_handler.get_status()

{'Shelf_1': None, 'Shelf_2': 'plate_2', 'Jubilee': 'plate_1'}

## Clear or Overwrite Occupancy

`clear_position()` marks a position as empty in software. Use it only when the physical setup matches that state.

In [ ]:
plate_handler.clear_position("Jubilee")
plate_handler.register_plate("plate_1", "Shelf_1", overwrite=True)

plate_handler.get_status()

{'Shelf_1': None, 'Shelf_2': 'plate_2', 'Jubilee': 'plate_1'}

## Exchange Two Plates

`exchange_plates()` swaps two registered plates. It needs one empty registered buffer position unless you pass a specific `buffer_position_id`.

In [31]:
plate_handler.exchange_plates(
    "plate_1",
    "plate_2",
    # buffer_position_id="Jubilee",
    move_speed=100,
)

plate_handler.get_status()

{'Shelf_1': None, 'Shelf_2': 'plate_1', 'Jubilee': 'plate_2'}

## Save Current Registry

After manually changing occupancy with `register_plate()` or `clear_position()`, use `save_positions()` if you want the JSON file to reflect the current software registry.

In [ ]:
plate_handler.save_positions(POSITION_FILE)

PosixPath('position.json')

[SDK][ERROR][2026-08-05 18:03:41][base.py:168] - - [report-socket] socket read timeout
[SDK][ERROR][2026-08-05 18:03:48][base.py:93] - - [main-socket] send error: [Errno 64] Host is down
[SDK][ERROR][2026-08-05 18:03:48][base.py:1342] - - report thread is break, connected=False, failed_cnts=2
